# A1 AMP Results — IID Test Set

This notebook is a thin presentation layer for the **canonical evaluator outputs** produced by `src/experiments/11_evaluate_amp.py`. It does not calculate F1, Jaccard, bootstrap intervals, thresholds, or predictions. The reference is the **SHERLOC Legacy Keywords silver reference**, not human-adjudicated gold labels.

The notebook reports incomplete artifacts as pending. It does not generate scientific conclusions or change the frozen protocol.

## 1. Setup and canonical-artifact contract

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

EXPECTED_METHODS = ("M1", "M2", "M3", "M4")


def locate_repo_root() -> Path:
    """Locate the repository without relying on the notebook launch directory."""
    configured = os.environ.get("SHERLOC_REPO_ROOT")
    starts = [Path(configured).expanduser()] if configured else []
    starts.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in starts:
        if (candidate / "src/experiments/11_evaluate_amp.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate SHERLOC_Case_Analysis. Start Jupyter in the repository "
        "or set SHERLOC_REPO_ROOT."
    )


REPO_ROOT = locate_repo_root()
METRICS_ROOT = REPO_ROOT / "outputs/metrics"


def load_json(path: Path) -> dict:
    if not path.is_file():
        display(Markdown(f"> **Pending:** `{path.relative_to(REPO_ROOT)}` does not exist."))
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def load_csv(path: Path, required_columns=()) -> pd.DataFrame:
    """Load an evaluator table, reporting absence without synthesizing results."""
    if not path.is_file():
        display(Markdown(f"> **Pending:** `{path.relative_to(REPO_ROOT)}` does not exist."))
        return pd.DataFrame(columns=list(required_columns))
    frame = pd.read_csv(path)
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing canonical columns: {sorted(missing)}")
    return frame


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def show_or_pending(frame: pd.DataFrame, message="No canonical rows are available yet."):
    if frame.empty:
        display(Markdown(f"> **Pending:** {message}"))
    else:
        display(frame)


def methods_complete(manifest: dict, evaluation: str) -> bool:
    methods = manifest.get("evaluations", {}).get(evaluation, {}).get("methods", [])
    return set(methods) == set(EXPECTED_METHODS)


manifest_path = METRICS_ROOT / "amp_evaluation_manifest.json"
evaluation_manifest = load_json(manifest_path)


## 2. Artifact availability and completion gate

In [ ]:
canonical_inputs = [
    METRICS_ROOT / "amp_evaluation_manifest.json",
    METRICS_ROOT / "a1/amp_primary_results.csv",
    METRICS_ROOT / "a1/amp_per_label.csv",
    METRICS_ROOT / "a1/amp_bootstrap_cis.csv",
    METRICS_ROOT / "a1/amp_case_level_errors.csv",
    METRICS_ROOT / "a2/amp_primary_results.csv",
    METRICS_ROOT / "a2/amp_per_fold.csv",
    METRICS_ROOT / "a2/amp_per_label.csv",
    METRICS_ROOT / "a2/amp_per_jurisdiction.csv",
    METRICS_ROOT / "a2/amp_bootstrap_cis.csv",
    METRICS_ROOT / "a2/amp_case_level_errors.csv",
    METRICS_ROOT / "amp_a1_to_a2_deltas.csv",
]
availability = pd.DataFrame(
    {
        "artifact": [str(path.relative_to(REPO_ROOT)) for path in canonical_inputs],
        "available": [path.is_file() for path in canonical_inputs],
    }
)
display(availability)

gate = evaluation_manifest.get("final_completion_gate", "PENDING")
complete = (
    gate == "PASSED_M1_M2_M3_M4_A1_A2"
    and methods_complete(evaluation_manifest, "A1")
    and methods_complete(evaluation_manifest, "A2")
)
if complete:
    display(Markdown("**Canonical completion gate: PASSED for M1-M4 in A1 and A2.**"))
else:
    display(Markdown(
        "> **Incomplete benchmark:** canonical M1-M4 A1/A2 outputs are not complete. "
        "Any available rows are technical previews, not the final comparison."
    ))


## 3. Frozen experiment metadata

In [ ]:
ontology_path = REPO_ROOT / "config/amp_ontology_v1.yaml"
demo_path = REPO_ROOT / "config/experiments/demo_bank_amp_v1.yaml"
llm_path = REPO_ROOT / "config/experiments/llm_extraction_amp_v2.yaml"
m1_config_path = REPO_ROOT / "config/experiments/m1_tfidf_logreg_amp_v2.yaml"
m2_config_path = REPO_ROOT / "config/experiments/m2_modernbert_amp_v2.yaml"

ontology = load_json(ontology_path)
demo_bank = load_json(demo_path)
llm_config = load_json(llm_path)
m1_config = load_json(m1_config_path)
m2_config = load_json(m2_config_path)

frozen_metadata = pd.DataFrame([
    {"item": "Primary cohort", "value": m1_config.get("primary_cohort_id", "PENDING")},
    {"item": "Reference terminology", "value": evaluation_manifest.get("reference_terminology", "PENDING")},
    {"item": "Ontology", "value": ontology.get("ontology_id", "PENDING")},
    {"item": "Ontology SHA-256", "value": sha256_file(ontology_path) if ontology_path.is_file() else "PENDING"},
    {"item": "Demo bank", "value": demo_bank.get("bank_id", "PENDING")},
    {"item": "Demo-bank SHA-256", "value": sha256_file(demo_path) if demo_path.is_file() else "PENDING"},
    {"item": "M3 prompt SHA-256", "value": llm_config.get("methods", {}).get("M3", {}).get("prompt_sha256", "PENDING")},
    {"item": "M4 prompt SHA-256", "value": llm_config.get("methods", {}).get("M4", {}).get("prompt_sha256", "PENDING")},
    {"item": "Bootstrap protocol", "value": json.dumps(evaluation_manifest.get("bootstrap", {}), sort_keys=True)},
])
display(frozen_metadata)


## 4. A1 split composition

In [ ]:
a1_split_path = REPO_ROOT / "data/splits/a1_iid_split_final_v1.csv"
a1_split = load_csv(a1_split_path, ("search_rank", "split", "effective_supervised_train"))
if not a1_split.empty:
    split_composition = (
        a1_split.groupby("split", dropna=False)
        .agg(cases=("search_rank", "size"), supervised_train=("effective_supervised_train", "sum"))
        .reset_index()
    )
    display(split_composition)
    display(pd.DataFrame([{"split_sha256": sha256_file(a1_split_path)}]))
else:
    show_or_pending(a1_split)


## 5. Frozen silver-reference label distribution by A1 role

In [ ]:
label_ids = [
    item["id"]
    for family in ("ACT", "MEANS", "PURPOSE")
    for item in ontology.get("families", {}).get(family, [])
]
if not a1_split.empty and label_ids:
    missing_labels = set(label_ids) - set(a1_split.columns)
    if missing_labels:
        raise ValueError(f"A1 split lacks ontology columns: {sorted(missing_labels)}")
    label_distribution = a1_split.groupby("split")[label_ids].sum().T
    label_distribution.index.name = "label_id"
    display(label_distribution)
else:
    display(Markdown("> **Pending:** frozen split or ontology is unavailable."))


## 6. M1 and M2 frozen configurations and validation-selected thresholds

In [ ]:
def model_run_summary(method: str) -> dict:
    metadata = load_json(REPO_ROOT / f"outputs/models/{method.lower()}/a1/run_metadata.json")
    selection = metadata.get("selection", {})
    return {
        "method": method,
        "status": metadata.get("status", "PENDING"),
        "run_id": metadata.get("run_id", "PENDING"),
        "selected_global_threshold": selection.get("selected_global_threshold", "PENDING"),
        "selected_hyperparameters": json.dumps(selection.get("selected_hyperparameters", {}), sort_keys=True),
        "selection_data": "VALIDATION_ONLY",
        "test_labels_used_for_selection": metadata.get("test_labels_used_for_selection", "PENDING"),
    }

display(pd.DataFrame([model_run_summary("M1"), model_run_summary("M2")]))


## 7. M3/M4 prompt and demonstration metadata

In [ ]:
llm_rows = []
for method in ("M3", "M4"):
    details = llm_config.get("methods", {}).get(method, {})
    llm_rows.append({
        "method": method,
        "experiment_id": details.get("experiment_id", "PENDING"),
        "prompt_version": details.get("prompt_version", "PENDING"),
        "prompt_sha256": details.get("prompt_sha256", "PENDING"),
        "demonstration_count": details.get("demonstration_count", "PENDING"),
        "model_requested": llm_config.get("api_request", {}).get("model", "PENDING"),
    })
display(pd.DataFrame(llm_rows))

a1_bank = llm_config.get("methods", {}).get("M4", {}).get("evaluation_banks", {}).get("A1", {})
display(pd.DataFrame([{
    "M4_A1_ordered_search_ranks": a1_bank.get("ordered_search_ranks", "PENDING"),
    "membership_sha256": a1_bank.get("membership_sha256", "PENDING"),
}]))


## 8. Canonical M1–M4 A1 comparison

In [ ]:
a1_primary = load_csv(
    METRICS_ROOT / "a1/amp_primary_results.csv",
    ("method", "macro_f1", "micro_f1", "exact_set_accuracy", "example_jaccard", "test_n"),
)
if not a1_primary.empty:
    order = {method: index for index, method in enumerate(EXPECTED_METHODS)}
    a1_primary = a1_primary.assign(_order=a1_primary["method"].map(order)).sort_values("_order").drop(columns="_order")
show_or_pending(a1_primary, "run the canonical evaluator after prediction artifacts are complete.")


## 9. Visual summaries of canonical aggregate metrics

In [ ]:
if not a1_primary.empty:
    columns = ["macro_f1", "micro_f1", "exact_set_accuracy", "example_jaccard"]
    axes = a1_primary.set_index("method")[columns].plot.bar(
        subplots=True, layout=(2, 2), figsize=(12, 8), legend=False, ylim=(0, 1),
        title=["Macro-F1", "Micro-F1", "Exact-set accuracy", "Example Jaccard"],
    )
    plt.suptitle("A1 canonical evaluator metrics (descriptive only)")
    plt.tight_layout()
else:
    display(Markdown("> **Pending:** no canonical A1 aggregate table to plot."))


## 10. Canonical per-label precision, recall, and F1

In [ ]:
a1_per_label = load_csv(
    METRICS_ROOT / "a1/amp_per_label.csv",
    ("method", "label_id", "family", "support", "precision", "recall", "f1", "status"),
)
show_or_pending(a1_per_label)


## 11. Canonical bootstrap confidence intervals

In [ ]:
a1_bootstrap = load_csv(
    METRICS_ROOT / "a1/amp_bootstrap_cis.csv",
    ("method", "metric", "estimate", "ci_lower", "ci_upper", "n_resamples", "seed"),
)
show_or_pending(a1_bootstrap)


## 12. Fixed 0.50-threshold sensitivity (M1/M2 only)

In [ ]:
threshold_sensitivity = load_csv(
    METRICS_ROOT / "amp_threshold_0_50_sensitivity.csv",
    ("method", "evaluation", "prediction_variant", "macro_f1", "micro_f1"),
)
a1_sensitivity = threshold_sensitivity.loc[threshold_sensitivity.get("evaluation", pd.Series(dtype=str)).eq("A1")] if not threshold_sensitivity.empty else threshold_sensitivity
show_or_pending(a1_sensitivity)


## 13. Technical observations for researcher review

Add descriptive, non-speculative notes here after the canonical completion gate passes. Do not use test-set observations to alter prompts, demonstrations, thresholds, preprocessing, architecture, or the frozen metric protocol. Paper-level Results/Discussion conclusions are intentionally not generated by this notebook.